In [ ]:
%%writefile milestone1_robust.cpp
#include <algorithm>
#include <cassert>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <random>
#include <set>
#include <sstream>
#include <string>
#include <unordered_set>
#include <utility>
#include <vector>

namespace pip {

constexpr double EPS = 1e-9;

struct Point {
    double x{};
    double y{};
};

struct BoundingBox {
    double minX{std::numeric_limits<double>::infinity()};
    double minY{std::numeric_limits<double>::infinity()};
    double maxX{-std::numeric_limits<double>::infinity()};
    double maxY{-std::numeric_limits<double>::infinity()};

    bool isValid() const {
        return minX <= maxX && minY <= maxY;
    }

    void expand(const Point& p) {
        minX = std::min(minX, p.x);
        minY = std::min(minY, p.y);
        maxX = std::max(maxX, p.x);
        maxY = std::max(maxY, p.y);
    }

    void expand(const BoundingBox& other) {
        if (!other.isValid()) return;
        minX = std::min(minX, other.minX);
        minY = std::min(minY, other.minY);
        maxX = std::max(maxX, other.maxX);
        maxY = std::max(maxY, other.maxY);
    }

    bool contains(const Point& p, double eps = EPS) const {
        return p.x >= minX - eps && p.x <= maxX + eps &&
               p.y >= minY - eps && p.y <= maxY + eps;
    }

    bool contains(const BoundingBox& other, double eps = EPS) const {
        return other.minX >= minX - eps && other.maxX <= maxX + eps &&
               other.minY >= minY - eps && other.maxY <= maxY + eps;
    }

    bool intersects(const BoundingBox& other, double eps = EPS) const {
        return !(other.minX > maxX + eps || other.maxX < minX - eps ||
                 other.minY > maxY + eps || other.maxY < minY - eps);
    }
};

using Ring = std::vector<Point>;

struct PolygonPart {
    Ring outer;
    std::vector<Ring> holes;
    BoundingBox bbox;
};

struct PolygonRegion {
    int id{};
    std::string name;
    std::vector<PolygonPart> parts;
    BoundingBox bbox;
};

enum class Location {
    Outside,
    Boundary,
    Inside
};

double cross(const Point& a, const Point& b, const Point& c) {
    return (b.x - a.x) * (c.y - a.y) - (b.y - a.y) * (c.x - a.x);
}

bool pointOnSegment(const Point& p, const Point& a, const Point& b, double eps = EPS) {
    if (std::fabs(cross(a, b, p)) > eps) return false;
    const double minX = std::min(a.x, b.x) - eps;
    const double maxX = std::max(a.x, b.x) + eps;
    const double minY = std::min(a.y, b.y) - eps;
    const double maxY = std::max(a.y, b.y) + eps;
    return p.x >= minX && p.x <= maxX && p.y >= minY && p.y <= maxY;
}

BoundingBox computeBoundingBox(const Ring& ring) {
    BoundingBox box;
    for (const auto& p : ring) box.expand(p);
    return box;
}

BoundingBox computeBoundingBox(const PolygonPart& part) {
    BoundingBox box = computeBoundingBox(part.outer);
    for (const auto& hole : part.holes) box.expand(computeBoundingBox(hole));
    return box;
}

BoundingBox computeBoundingBox(const PolygonRegion& region) {
    BoundingBox box;
    for (const auto& part : region.parts) box.expand(part.bbox);
    return box;
}

void finalizePolygonPart(PolygonPart& part) {
    part.bbox = computeBoundingBox(part);
}

void finalizePolygonRegion(PolygonRegion& region) {
    for (auto& part : region.parts) {
        finalizePolygonPart(part);
    }
    region.bbox = computeBoundingBox(region);
}

Location pointInRing(const Point& p, const Ring& ring) {
    if (ring.size() < 3) return Location::Outside;

    bool inside = false;
    const size_t n = ring.size();
    for (size_t i = 0, j = n - 1; i < n; j = i++) {
        const Point& a = ring[j];
        const Point& b = ring[i];

        if (pointOnSegment(p, a, b)) {
            return Location::Boundary;
        }

        const bool straddles = (a.y > p.y) != (b.y > p.y);
        if (straddles) {
            const double xIntersect = a.x + (b.x - a.x) * (p.y - a.y) / (b.y - a.y);
            if (xIntersect > p.x + EPS) {
                inside = !inside;
            } else if (std::fabs(xIntersect - p.x) <= EPS) {
                return Location::Boundary;
            }
        }
    }
    return inside ? Location::Inside : Location::Outside;
}

Location pointInPolygonPart(const Point& p, const PolygonPart& part) {
    if (!part.bbox.contains(p)) return Location::Outside;

    const Location outerLoc = pointInRing(p, part.outer);
    if (outerLoc == Location::Outside) return Location::Outside;
    if (outerLoc == Location::Boundary) return Location::Boundary;

    for (const auto& hole : part.holes) {
        const BoundingBox holeBox = computeBoundingBox(hole);
        if (!holeBox.contains(p)) continue;
        const Location holeLoc = pointInRing(p, hole);
        if (holeLoc == Location::Boundary) {
            return Location::Boundary; // boundary-inclusive semantics
        }
        if (holeLoc == Location::Inside) {
            return Location::Outside; // strictly inside a hole
        }
    }

    return Location::Inside;
}

Location pointInPolygonRegion(const Point& p, const PolygonRegion& region) {
    if (!region.bbox.contains(p)) return Location::Outside;

    bool foundInside = false;
    for (const auto& part : region.parts) {
        if (!part.bbox.contains(p)) continue;
        const Location loc = pointInPolygonPart(p, part);
        if (loc == Location::Boundary) return Location::Boundary;
        if (loc == Location::Inside) foundInside = true;
    }
    return foundInside ? Location::Inside : Location::Outside;
}

class Quadtree {
public:
    explicit Quadtree(BoundingBox boundary,
                      size_t capacity = 8,
                      int maxDepth = 12)
        : root_(std::move(boundary), capacity, maxDepth, 0) {}

    void build(const std::vector<PolygonRegion>& regions) {
        for (const auto& region : regions) {
            root_.insert(&region);
        }
    }

    void query(const Point& p, std::vector<const PolygonRegion*>& out) const {
        out.clear();
        root_.query(p, out);
    }

private:
    struct Node {
        BoundingBox boundary;
        size_t capacity;
        int maxDepth;
        int depth;
        bool divided{false};
        std::vector<const PolygonRegion*> items;
        std::vector<Node> children;

        Node(BoundingBox box, size_t cap, int maxD, int dep)
            : boundary(std::move(box)), capacity(cap), maxDepth(maxD), depth(dep) {}

        void subdivide() {
            if (divided) return;
            const double midX = (boundary.minX + boundary.maxX) * 0.5;
            const double midY = (boundary.minY + boundary.maxY) * 0.5;

            children.reserve(4);
            children.emplace_back(BoundingBox{boundary.minX, midY, midX, boundary.maxY}, capacity, maxDepth, depth + 1); // NW
            children.emplace_back(BoundingBox{midX, midY, boundary.maxX, boundary.maxY}, capacity, maxDepth, depth + 1); // NE
            children.emplace_back(BoundingBox{boundary.minX, boundary.minY, midX, midY}, capacity, maxDepth, depth + 1); // SW
            children.emplace_back(BoundingBox{midX, boundary.minY, boundary.maxX, midY}, capacity, maxDepth, depth + 1); // SE
            divided = true;
        }

        int childIndexIfFullyContained(const BoundingBox& box) const {
            if (!divided) return -1;
            for (int i = 0; i < 4; ++i) {
                if (children[i].boundary.contains(box)) return i;
            }
            return -1;
        }

        void maybePushDownExistingItems() {
            if (!divided) return;
            std::vector<const PolygonRegion*> remaining;
            remaining.reserve(items.size());
            for (const auto* item : items) {
                const int idx = childIndexIfFullyContained(item->bbox);
                if (idx >= 0) {
                    children[idx].insert(item);
                } else {
                    remaining.push_back(item);
                }
            }
            items.swap(remaining);
        }

        bool insert(const PolygonRegion* region) {
            if (!boundary.intersects(region->bbox)) return false;

            if (divided) {
                const int idx = childIndexIfFullyContained(region->bbox);
                if (idx >= 0) {
                    return children[idx].insert(region);
                }
            }

            if (items.size() < capacity || depth >= maxDepth) {
                items.push_back(region);
                return true;
            }

            if (!divided) {
                subdivide();
                maybePushDownExistingItems();
            }

            const int idx = childIndexIfFullyContained(region->bbox);
            if (idx >= 0) {
                return children[idx].insert(region);
            }

            items.push_back(region); // spanning region stays at current node
            return true;
        }

        void query(const Point& p, std::vector<const PolygonRegion*>& out) const {
            if (!boundary.contains(p)) return;

            for (const auto* region : items) {
                if (region->bbox.contains(p)) out.push_back(region);
            }

            if (!divided) return;
            for (const auto& child : children) {
                if (child.boundary.contains(p)) {
                    child.query(p, out);
                }
            }
        }
    };

    Node root_;
};

std::vector<Point> generateUniformPoints(size_t n,
                                         const BoundingBox& domain,
                                         std::mt19937_64& rng) {
    std::uniform_real_distribution<double> dx(domain.minX, domain.maxX);
    std::uniform_real_distribution<double> dy(domain.minY, domain.maxY);
    std::vector<Point> points;
    points.reserve(n);
    for (size_t i = 0; i < n; ++i) {
        points.push_back({dx(rng), dy(rng)});
    }
    return points;
}

std::vector<Point> generateClusteredPoints(size_t n,
                                           const BoundingBox& domain,
                                           std::mt19937_64& rng) {
    std::vector<Point> points;
    points.reserve(n);

    std::vector<Point> centers = {
        {25.0, 25.0}, {50.0, 50.0}, {75.0, 70.0}, {35.0, 80.0}
    };
    std::discrete_distribution<int> chooseCenter({35, 30, 20, 15});
    std::normal_distribution<double> noise(0.0, 6.0);
    std::uniform_real_distribution<double> mix(0.0, 1.0);
    std::uniform_real_distribution<double> ux(domain.minX, domain.maxX);
    std::uniform_real_distribution<double> uy(domain.minY, domain.maxY);

    auto clamp = [](double value, double lo, double hi) {
        return std::max(lo, std::min(value, hi));
    };

    for (size_t i = 0; i < n; ++i) {
        if (mix(rng) < 0.80) {
            const Point c = centers[chooseCenter(rng)];
            points.push_back({
                clamp(c.x + noise(rng), domain.minX, domain.maxX),
                clamp(c.y + noise(rng), domain.minY, domain.maxY)
            });
        } else {
            points.push_back({ux(rng), uy(rng)});
        }
    }
    return points;
}

std::vector<int> classifyPointBruteForce(const Point& p,
                                         const std::vector<PolygonRegion>& regions) {
    std::vector<int> matches;
    for (const auto& region : regions) {
        const Location loc = pointInPolygonRegion(p, region);
        if (loc != Location::Outside) {
            matches.push_back(region.id);
        }
    }
    std::sort(matches.begin(), matches.end());
    return matches;
}

std::vector<int> classifyPointIndexed(const Point& p,
                                      const Quadtree& index,
                                      std::vector<const PolygonRegion*>& scratch) {
    index.query(p, scratch);
    std::vector<int> matches;
    matches.reserve(scratch.size());
    std::unordered_set<int> seen;
    for (const auto* region : scratch) {
        if (!seen.insert(region->id).second) continue;
        const Location loc = pointInPolygonRegion(p, *region);
        if (loc != Location::Outside) {
            matches.push_back(region->id);
        }
    }
    std::sort(matches.begin(), matches.end());
    return matches;
}

struct BenchmarkStats {
    double bruteForceSeconds{};
    double quadtreeSeconds{};
    double averageCandidates{};
    size_t totalMatches{};
};

BenchmarkStats benchmarkDataset(const std::vector<Point>& points,
                                const std::vector<PolygonRegion>& regions,
                                const Quadtree& index,
                                bool verifyExact) {
    using clock = std::chrono::high_resolution_clock;

    size_t totalMatchesBF = 0;
    auto startBF = clock::now();
    std::vector<std::vector<int>> bruteResults;
    if (verifyExact) bruteResults.reserve(points.size());

    for (const auto& p : points) {
        auto result = classifyPointBruteForce(p, regions);
        totalMatchesBF += result.size();
        if (verifyExact) bruteResults.push_back(std::move(result));
    }
    auto endBF = clock::now();

    size_t totalMatchesQT = 0;
    size_t totalCandidates = 0;
    std::vector<const PolygonRegion*> scratch;
    auto startQT = clock::now();
    for (size_t i = 0; i < points.size(); ++i) {
        index.query(points[i], scratch);
        totalCandidates += scratch.size();
        auto result = classifyPointIndexed(points[i], index, scratch);
        totalMatchesQT += result.size();
        if (verifyExact && result != bruteResults[i]) {
            std::ostringstream oss;
            oss << "Mismatch at point (" << points[i].x << ", " << points[i].y << ")";
            throw std::runtime_error(oss.str());
        }
    }
    auto endQT = clock::now();

    if (totalMatchesBF != totalMatchesQT) {
        throw std::runtime_error("Mismatch in total match counts between brute force and quadtree.");
    }

    BenchmarkStats stats;
    stats.bruteForceSeconds = std::chrono::duration<double>(endBF - startBF).count();
    stats.quadtreeSeconds = std::chrono::duration<double>(endQT - startQT).count();
    stats.averageCandidates = points.empty() ? 0.0 : static_cast<double>(totalCandidates) / static_cast<double>(points.size());
    stats.totalMatches = totalMatchesQT;
    return stats;
}

PolygonPart makeRectangle(double minX, double minY, double maxX, double maxY) {
    PolygonPart part;
    part.outer = {{minX, minY}, {maxX, minY}, {maxX, maxY}, {minX, maxY}};
    finalizePolygonPart(part);
    return part;
}

PolygonRegion makeSimpleRectangleRegion(int id, const std::string& name,
                                        double minX, double minY, double maxX, double maxY) {
    PolygonRegion region;
    region.id = id;
    region.name = name;
    region.parts.push_back(makeRectangle(minX, minY, maxX, maxY));
    finalizePolygonRegion(region);
    return region;
}

std::vector<PolygonRegion> buildCorrectnessDataset() {
    std::vector<PolygonRegion> regions;

    // Region 1: simple square
    regions.push_back(makeSimpleRectangleRegion(1, "Square-A", 10.0, 10.0, 30.0, 30.0));

    // Region 2: donut polygon with an interior hole
    PolygonRegion donut;
    donut.id = 2;
    donut.name = "Donut-Zone";
    PolygonPart donutPart;
    donutPart.outer = {{40.0, 40.0}, {80.0, 40.0}, {80.0, 80.0}, {40.0, 80.0}};
    donutPart.holes.push_back({{50.0, 50.0}, {70.0, 50.0}, {70.0, 70.0}, {50.0, 70.0}});
    finalizePolygonPart(donutPart);
    donut.parts.push_back(donutPart);
    finalizePolygonRegion(donut);
    regions.push_back(donut);

    // Region 3: multipolygon with two disjoint parts
    PolygonRegion multi;
    multi.id = 3;
    multi.name = "Multi-Part";
    multi.parts.push_back(makeRectangle(5.0, 60.0, 15.0, 70.0));
    multi.parts.push_back(makeRectangle(20.0, 75.0, 35.0, 90.0));
    finalizePolygonRegion(multi);
    regions.push_back(multi);

    // Region 4: overlapping square to test multiple memberships
    regions.push_back(makeSimpleRectangleRegion(4, "Overlap-Zone", 25.0, 25.0, 45.0, 45.0));

    return regions;
}

std::vector<PolygonRegion> buildBenchmarkDataset(int gridX, int gridY) {
    std::vector<PolygonRegion> regions;
    int id = 1000;

    const double stepX = 100.0 / static_cast<double>(gridX);
    const double stepY = 100.0 / static_cast<double>(gridY);

    for (int ix = 0; ix < gridX; ++ix) {
        for (int iy = 0; iy < gridY; ++iy) {
            const double minX = ix * stepX;
            const double minY = iy * stepY;
            const double maxX = minX + stepX * 0.9;
            const double maxY = minY + stepY * 0.9;
            regions.push_back(makeSimpleRectangleRegion(id++, "Grid-Cell", minX, minY, maxX, maxY));
        }
    }

    // Add a few larger polygons with holes and multipart regions for realism.
    PolygonRegion campus;
    campus.id = id++;
    campus.name = "Campus-Ring";
    PolygonPart campusPart;
    campusPart.outer = {{15, 15}, {55, 15}, {55, 55}, {15, 55}};
    campusPart.holes.push_back({{28, 28}, {42, 28}, {42, 42}, {28, 42}});
    finalizePolygonPart(campusPart);
    campus.parts.push_back(campusPart);
    finalizePolygonRegion(campus);
    regions.push_back(campus);

    PolygonRegion islands;
    islands.id = id++;
    islands.name = "Twin-Islands";
    islands.parts.push_back(makeRectangle(62, 18, 72, 28));
    islands.parts.push_back(makeRectangle(76, 20, 88, 34));
    finalizePolygonRegion(islands);
    regions.push_back(islands);

    return regions;
}

std::string vecToString(const std::vector<int>& v) {
    std::ostringstream oss;
    oss << "{";
    for (size_t i = 0; i < v.size(); ++i) {
        if (i) oss << ", ";
        oss << v[i];
    }
    oss << "}";
    return oss.str();
}

void runDeterministicCorrectnessTests(const std::vector<PolygonRegion>& regions) {
    const BoundingBox domain{0.0, 0.0, 100.0, 100.0};
    Quadtree index(domain, 6, 10);
    index.build(regions);
    std::vector<const PolygonRegion*> scratch;

    struct TestCase {
        Point p;
        std::vector<int> expected;
        std::string label;
    };

    const std::vector<TestCase> tests = {
        {{10, 10}, {1}, "square vertex"},
        {{20, 20}, {1}, "square interior"},
        {{30, 30}, {1, 4}, "shared overlap boundary"},
        {{35, 35}, {4}, "overlap-only interior"},
        {{60, 60}, {}, "inside hole => outside polygon"},
        {{50, 60}, {2}, "hole boundary (included as boundary)"},
        {{45, 45}, {2, 4}, "donut outer boundary and overlap corner"},
        {{12, 65}, {3}, "multipart first island"},
        {{25, 80}, {3}, "multipart second island"},
        {{17, 72}, {}, "outside multipart"},
        {{90, 90}, {}, "outside all"}
    };

    for (const auto& test : tests) {
        const auto brute = classifyPointBruteForce(test.p, regions);
        const auto indexed = classifyPointIndexed(test.p, index, scratch);
        if (brute != test.expected || indexed != test.expected) {
            std::cerr << "Deterministic test failed: " << test.label << "\n"
                      << "Expected: " << vecToString(test.expected) << "\n"
                      << "Brute:    " << vecToString(brute) << "\n"
                      << "Indexed:  " << vecToString(indexed) << "\n";
            throw std::runtime_error("Deterministic correctness suite failed.");
        }
    }
}

void printBenchmarkLine(const std::string& label, const BenchmarkStats& stats) {
    std::cout << std::left << std::setw(18) << label
              << " | Brute Force: " << std::setw(8) << std::fixed << std::setprecision(4) << stats.bruteForceSeconds << " s"
              << " | Quadtree: " << std::setw(8) << stats.quadtreeSeconds << " s"
              << " | Avg candidates/pt: " << std::setw(6) << std::setprecision(2) << stats.averageCandidates
              << " | Total matches: " << stats.totalMatches
              << '\n';
}

} // namespace pip

int main() {
    using namespace pip;

    try {
        std::cout << "=== Milestone 1: Sequential Baseline with Robust Spatial Indexing ===\n\n";

        const BoundingBox domain{0.0, 0.0, 100.0, 100.0};

        // 1) Correctness dataset: edges, vertices, holes, multipolygons, overlaps
        auto correctnessRegions = buildCorrectnessDataset();
        runDeterministicCorrectnessTests(correctnessRegions);
        std::cout << "[OK] Deterministic correctness tests passed (edges, vertices, holes, multipolygons, overlaps).\n";

        // 2) Random validation: exact brute-force vs quadtree agreement
        Quadtree correctnessIndex(domain, 6, 10);
        correctnessIndex.build(correctnessRegions);

        std::mt19937_64 rng(42);
        auto uniformValidation = generateUniformPoints(50000, domain, rng);
        auto clusteredValidation = generateClusteredPoints(50000, domain, rng);

        const auto uniformValidationStats = benchmarkDataset(uniformValidation, correctnessRegions, correctnessIndex, true);
        const auto clusteredValidationStats = benchmarkDataset(clusteredValidation, correctnessRegions, correctnessIndex, true);
        std::cout << "[OK] Random validation passed for uniform and clustered GPS datasets.\n\n";

        std::cout << "Validation summary:\n";
        printBenchmarkLine("Uniform (50k)", uniformValidationStats);
        printBenchmarkLine("Clustered (50k)", clusteredValidationStats);
        std::cout << '\n';

        // 3) Baseline benchmark with a larger polygon dataset
        auto benchmarkRegions = buildBenchmarkDataset(20, 20); // 400+ regions
        Quadtree benchmarkIndex(domain, 8, 12);
        benchmarkIndex.build(benchmarkRegions);

        auto uniformBenchmark = generateUniformPoints(200000, domain, rng);
        auto clusteredBenchmark = generateClusteredPoints(200000, domain, rng);

        const auto uniformStats = benchmarkDataset(uniformBenchmark, benchmarkRegions, benchmarkIndex, true);
        const auto clusteredStats = benchmarkDataset(clusteredBenchmark, benchmarkRegions, benchmarkIndex, true);

        std::cout << "Baseline benchmark:\n";
        printBenchmarkLine("Uniform (200k)", uniformStats);
        printBenchmarkLine("Clustered (200k)", clusteredStats);

        std::cout << "\nMilestone 1 implementation complete: robust ray-casting, holes, multipolygons, quadtree MBB filtering, synthetic GPS generation, and brute-force verification.\n";
    } catch (const std::exception& ex) {
        std::cerr << "ERROR: " << ex.what() << '\n';
        return 1;
    }

    return 0;
}


In [ ]:
!g++ -std=c++17 -O3 milestone1_robust.cpp -o milestone1_robust


In [ ]:
!./milestone1_robust
